# RFM in Action: Behavioral Segmentation for E-Commerce
### Phase 1 — Data Cleaning & Validation


**Goal:** Validate data integrity, enforce consistent transaction rules, and produce a clean transactional table suitable for customer-level aggregation.

**Business Question:**
How can an online retailer identify high-value customers, detect churn risk early, and design targeted retention strategies using historical transaction data?

**Dataset:** UCI Online Retail II dataset containing transactional purchase records for a UK-based online gift retailer (Dec 2009 – Dec 2011).

In [1]:
# setup
import pandas as pd
import numpy as np
import datetime as dt
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("online_retail_II.csv")

In [2]:
# preview data
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
print(df.shape)
df.info()

(1067371, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  object 
 1   StockCode    1067371 non-null  object 
 2   Description  1062989 non-null  object 
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  object 
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 65.1+ MB


### Observations:

- `InvoiceDate` col is of type 'object', however it contains datetime information, so we need to convert the column.
- Most cols have 1,067,371 rows with the exceptions of `Description` and `Customer ID` with 1,062,989 and 824,364 rows, respectively, indicating missing values. Since `Customer ID` is critical to our analysis, we must exclude the rows with missing customer IDs.
- `Customer ID` col is of type 'float64', but in this case, 'int64' is more efficient and semantically correct for this data.
- The data should be sorted chronologically.

In [4]:
# cleaning data based on above observations
df['InvoiceDatetime'] = pd.to_datetime(df['InvoiceDate'])
df = df[df['Customer ID'].notna()]
df['Customer ID'] = df['Customer ID'].astype(np.int64)
df = df.sort_values('InvoiceDatetime')
df = df.loc[:, ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDatetime', 'Price', 'Customer ID', 'Country']]

In [5]:
# examine numeric data
print(df.describe())

# count number of negative prices and quantities
print(f"Number of negative vals in Price: {len(df[df['Price'] < 0])}")
print(f"Number of negative vals in Quantity: {len(df[df['Quantity'] < 0])}")

# count number of duplicate rows
print(f"Number of duplicates: {df.duplicated().sum()}")

            Quantity                InvoiceDatetime          Price  \
count  824364.000000                         824364  824364.000000   
mean       12.414574  2011-01-01 22:29:28.042054144       3.676800   
min    -80995.000000            2009-12-01 07:45:00       0.000000   
25%         2.000000            2010-07-06 11:58:00       1.250000   
50%         5.000000            2010-12-03 14:26:00       1.950000   
75%        12.000000            2011-07-27 15:14:00       3.750000   
max     80995.000000            2011-12-09 12:50:00   38970.000000   
std       188.976099                            NaN      70.241388   

         Customer ID  
count  824364.000000  
mean    15324.638504  
min     12346.000000  
25%     13975.000000  
50%     15255.000000  
75%     16797.000000  
max     18287.000000  
std      1697.464450  
Number of negative vals in Price: 0
Number of negative vals in Quantity: 18744
Number of duplicates: 26479


### Observations:

- The min and max values of `InvoiceDatetime` indicate that the dataset spans customer transactions from December 2009 to December 2011.
- There are no negative prices detected, but there are negative quantities present. These negative values in the `Quantity` col likely represent returns/cancellations. For the sake of our customer segmentation and retention analysis, we will exclude transactions with non-positive quantities and prices to avoid understating customer purchasing behavior.
- According to the dataset's data card, `Invoice` numbers starting with the letter 'C' indicate cancellations, so we will exclude these from our analysis.
- There are zero-price transactions present, likely representing free samples or promotional items. We will exclude these transactions from revenue calculations for simplicity.
- There are 26,479 duplicate rows, which we need to remove from the dataset.

In [6]:
# drop rows where Price <= 0
df.drop(df[df['Price'] <= 0].index, inplace = True)

# drop rows where Quantity < 0
df.drop(df[df['Quantity'] <= 0].index, inplace = True)

# unselect rows where Invoice col contains 'C'0
df = df[~df['Invoice'].str.contains('C', na = False)]

# remove duplicates
df = df.drop_duplicates(keep = 'first')
print(f"Number of duplicates after cleaning: {df.duplicated().sum()}")

Number of duplicates after cleaning: 0


In [7]:
df.info()
# df.to_csv('transactions_clean.csv')

<class 'pandas.core.frame.DataFrame'>
Index: 779425 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   Invoice          779425 non-null  object        
 1   StockCode        779425 non-null  object        
 2   Description      779425 non-null  object        
 3   Quantity         779425 non-null  int64         
 4   InvoiceDatetime  779425 non-null  datetime64[ns]
 5   Price            779425 non-null  float64       
 6   Customer ID      779425 non-null  int64         
 7   Country          779425 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 53.5+ MB


## Final Dataset:

After applying data validation and cleaning rules, the final dataset contains **779,425** transaction records with complete customer identifiers and valid purchase activity. 

Final schema:
- 8 columns
- Proper datetime typing for transactional timestamps
- Integer customer identifiers
- No duplicate rows
- No missing values in critical fields
- Negative quantity and zero-price transactions excluded
- Cancellations excluded
- Transactions sorted chronologically
- Time coverage: December 2009 to December 2011

This communicates completeness, correctness, and readiness for downstream analysis.